# Otto Scents: YOLOv8 Training (Letter Identities A, B, C, D)
This notebook is specifically configured for your new dataset: **perfume-letters**.

### 1. Install Requirements

In [ ]:
!pip install ultralytics roboflow

### 2. Download Dataset from Roboflow
Using your specific project: `perfume-letters` version 1.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="9oRKwXj2MdugvomIxwFW")
project = rf.workspace("jc-jade").project("perfume-letters")
version = project.version(1)
dataset = version.download("yolov8")

### 3. Train the Model
Training YOLOv8 Nano to detect A, B, C, D bottle caps.

In [ ]:
from ultralytics import YOLO
import os

model = YOLO('yolov8n.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    plots=True
)

### 4. Validation: Test with Your Letter Images
Upload your new 'perfume-letters' test images here to verify the zoning logic (A=Zone1, B=Zone2, etc).

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
from google.colab import files
import os

SHELF_ZONES = {
    "A": {"x_min": 0.00, "x_max": 0.25},
    "B": {"x_min": 0.25, "x_max": 0.50},
    "C": {"x_min": 0.50, "x_max": 0.75},
    "D": {"x_min": 0.75, "x_max": 1.00}
}

def test_letters(image_path):
    model = YOLO('runs/detect/train/weights/best.pt')
    results = model.predict(source=image_path, conf=0.25)
    
    counts = {code: 0 for code in SHELF_ZONES.keys()}
    img_width = results[0].orig_shape[1]
    
    print(f"\n--- Detection Report for {os.path.basename(image_path)} ---")
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        
        # Clean class name to match A, B, C, D
        code = cls_name.upper().strip()
        
        if code in counts: 
            counts[code] += 1
            x1, y1, x2, y2 = box.xyxy[0]
            center_x_norm = ((x1 + x2) / 2) / img_width
            
            zone = SHELF_ZONES.get(code)
            if zone and not (zone["x_min"] <= center_x_norm <= zone["x_max"]):
                print(f"[!] MISPLACED: Product {code} found outside its zone!")
    
    for code, count in counts.items():
        print(f"Product {code}: {count} detected")
    
    cv2_imshow(results[0].plot())

print("Upload your new letter images to test:")
uploaded = files.upload()
for fn in uploaded.keys():
    test_letters(fn)

### 5. Export and Download Weights
Download `best.pt` for your Ubuntu VM.

In [ ]:
from google.colab import files
best_model = YOLO('runs/detect/train/weights/best.pt')
best_model.export(format='onnx')

files.download('runs/detect/train/weights/best.pt')